# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, where (in z, um) real tissue signal ends, using one already-finished round (default: `cells`), then estimates the time/data savings from trimming future rounds to that depth plus a small margin.

Part of the `multi_z` before_imaging pipeline: run after that pipeline's notebook 01 (cells hal_config) and notebook 03 (positions), against the finished `cells` round. The margin/savings estimate, trimmed-depth verification mosaic, and the per-FOV z table notebook 04 (`create_hal_config_and_shutters_multi_z.ipynb`) actually reads live in the follow-on notebook `09_multi_z_margin_export.ipynb` -- this notebook builds the state (`config`, `elevation_matrices`, `results_df`, ...) that one re-derives from its own cache. Lives in `after_imaging/` rather than `before_imaging/multi_z/` itself since it runs mid-acquisition, once the cells round has actually been imaged. Follows the architecture rules in [`NOTEBOOK_GUIDELINES.md`](../../NOTEBOOK_GUIDELINES.md) (repo root) throughout -- every nontrivial step below is a **calculation cell** (cached under `analysis/cache/measure_tissue_thickness/`, with `ProgressReporter` progress, skipping recomputation when its cache still matches the current inputs) followed by a separate **display cell** (plot or printed summary).

**Per-pixel tissue-elevation approach** (promoted from `notebooks/tests/tissue_thickness/01_elevation_heatmap.ipynb`'s own investigation -- see that notebook's own docstring/Review note for the full algorithm rationale, and `notebooks/tests/calculate_ffc/01_compare_ffc_methods.ipynb` for why "min" is the default FFC projection statistic). Replaces an earlier per-FOV Counter/true-pixel-count scalar approach: instead of one intensity histogram per FOV, every downsampled pixel gets its own "topmost foreground z" (an elevation map), and each FOV's scalar depth (`z_last_um`, what `before_imaging/multi_z`'s notebook 04 actually needs) is just that map's own per-FOV maximum. The library functions behind every step below live in `MERci.analysis.elevation`.

Procedure:
1. Resolve the target round's frame table and `CHANNEL_NM`'s z-steps.
2. Identify boundary (exterior-grid) vs. interior FOVs, plus each FOV's integer grid index (`identify_boundary_fovs`).
3. Build a flat-field-correction (FFC) field from every INTERIOR FOV's own full-z-stack projection (default statistic: `min` -- boundary FOVs are not reliably tissue-free, see the Review note in `01_elevation_heatmap.ipynb`). Has a **SLURM array option** (heaviest read step for this section: one full z-stack per interior FOV).
4. Estimate a background/foreground `THRESHOLD` from the boundary FOVs' own mid-z frame, in the FFC-corrected + downsampled space. Review the histogram plot and override `THRESHOLD` manually if it looks wrong.
5. For every real FOV in the round (the full grid, not a representative subset) and every z-plane: FFC-correct, downsample, threshold, and record the topmost foreground z per pixel (`compute_fov_elevation`) -- the heaviest read step in the notebook, with its own **SLURM array option**.
6. Derive each FOV's scalar `z_last_um` (that FOV's own elevation matrix maximum; `NaN` = no signal at all) and stitch every FOV's elevation matrix into one grid-indexed heatmap (`create_elevation_heatmap`), displayed next to a histogram of the same heatmap's own elevation values.
7. Render a static single-z mosaic at one representative depth (`Z_MOSAIC_UM`, default 25 um -- `create_z_mosaic`), with the same shared intensity scale, z label, and physical scale bar as the GIF below.
8. Assemble a z-sweep GIF of the same FFC-corrected, downsampled stacks (`create_gif`), with a shared intensity scale, a per-frame z label, and a physical scale bar.
9. Margin/savings estimation, the trimmed-depth verification mosaic, and the z-table export `before_imaging/multi_z`'s notebook 04 needs continue in the follow-on notebook `09_multi_z_margin_export.ipynb` -- split out because those three steps are `multi_z`-specific, unlike steps 1-8 above (useful for any pipeline that runs this tissue-thickness measurement).

## 1 — Setup

In [ ]:
import os
import sys
import json
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from skimage.transform import resize as sk_resize

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import read_image_frames, iter_image_frames
from MERci.acquisition.configs import (
    find_frame_table_for_hal_config, read_hal_exposure_time, get_fov_geometry, get_color_frame_indices,
)
from MERci.acquisition.merlin_config import load_microscope_orientation, apply_microscope_orientation
from MERci.analysis.elevation import (
    identify_boundary_fovs, compute_fov_projection, calculate_ffc, ffc_correct_and_downsample,
    estimate_background_threshold, compute_fov_elevation, create_elevation_heatmap,
    create_z_mosaic, create_gif, create_movie,
)
from MERci.analysis.ffc        import save_ffc_field, load_ffc_field
from MERci.analysis.fov        import _atomic_save
from MERci.analysis.round      import create_mosaic
from MERci.visualization       import display_mosaic, get_merci_figures_dir
from MERci.scheduler           import resolve_round_flip_y

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which microscope this experiment was acquired on (+ optional objective
# override -- None uses that microscope's own default objective).
MICROSCOPE = "ST2"
OBJECTIVE  = None

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# Per-pixel downsample factor shared by FFC correction, the elevation
# heatmap, and the z-sweep GIF (2304 / 16 = 144 px) -- see
# notebooks/tests/downsample_mosaic/01_compare_downsample_mosaic.ipynb's
# own visual comparison across factors.
DOWNSAMPLE_FACTOR = 16

# FFC field: which per-pixel z-projection statistic to pool across every
# interior FOV -- "min" (default, recommended) per notebooks/tests/
# calculate_ffc/01_compare_ffc_methods.ipynb's own real comparison; other
# options: "max", "median", "mean".
FFC_METHOD               = "min"
FFC_SMOOTH_SIGMA_PX      = 50.0   # 0 = no smoothing -- see that notebook's own Discussion
FFC_NORMALIZE_PERCENTILE = 99.99
FFC_MIN_VALUE            = 0.10

# Background/foreground THRESHOLD estimation (FFC-corrected + downsampled
# space): the highest pixel value (at BACKGROUND_PERCENTILE) observed among
# the N_BACKGROUND_FRAMES lowest-mean boundary FOVs.
N_BACKGROUND_FRAMES   = 20
BACKGROUND_PERCENTILE = 100.0

# Binarization intensity threshold; None = auto-estimate (see the
# background/foreground threshold section below) -- review that plot
# before trusting the estimate on a new experiment.
THRESHOLD = None

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Single-z mosaic (a static snapshot at one representative depth, same
# FFC-corrected/downsampled/scale-bar/z-label convention as the GIF).
Z_MOSAIC_UM = 25.0

# z-sweep GIF.
GIF_Z_STRIDE          = 1        # every Nth z-plane (1 = every frame)
GIF_FRAME_DURATION_MS = 300
GIF_SCALEBAR_UM       = 1000.0   # physical scale-bar length (1000 um = 1 mm)
GIF_PERCENTILE_CLIP   = (1.0, 99.0)

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- matplotlib's default
# sizes shrink relative to figsize, so a wide/short figure reads noticeably
# smaller than a square one at the same nominal size.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Positions tag       : {POSITIONS_TAG}")
print(f"Microscope         : {MICROSCOPE}  (objective override: {OBJECTIVE})")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")
print(f"Downsample factor  : {DOWNSAMPLE_FACTOR}")
print(f"FFC method         : {FFC_METHOD}")

In [ ]:
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
    pixel_size_um  = pixel_size_um,
    image_size_px  = image_size_px,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)

NOTEBOOK_NAME = "measure_tissue_thickness"
figures_dir = get_merci_figures_dir(SAMPLE_DIR, "after_imaging", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)

# NOTEBOOK_GUIDELINES.md #2/#3: every calculation cell below caches its result
# under analysis/cache/<notebook_name>/ and skips recomputation when a valid
# cache is already there.
cache_dir = config.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir.mkdir(parents=True, exist_ok=True)

# This repo's own bundled MERlin microscope-parameters JSONs -- resolve once,
# apply to every raw frame read below (a raw camera frame does not match the
# real stage layout otherwise, see MERci.acquisition.merlin_config).
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Step size: {config.step_size_um:.2f} um   image_size_px: {config.image_size_px}")
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)
round_info  = meta.rounds[target_round_id]

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

# [(0-based frame index, z um), ...], ascending z -- shared by every step below,
# plus the same two lists split out separately (frame_idx_list/z_um_list) for
# the functions in MERci.analysis.elevation that take them as separate args.
z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))
frame_idx_list  = [idx for idx, _ in z_frame_indices]
z_um_list       = [z for _, z in z_frame_indices]
mid_frame_idx   = get_color_frame_indices(frame_table)[CHANNEL_NM]

# {fov_id: (x, y)} -- scoped to this round's own real imaged FOVs (not the
# raw experiment-wide positions.txt), so transit-only FOVs never enter it.
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

print(f"Target round : {target_round_id}  ({len(positions)} FOV(s) with real files)")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Identify boundary FOVs

"Boundary FOVs" = the *exterior* FOVs of the imaged grid (outer perimeter + any hole edges) -- `identify_boundary_fovs` (`MERci.acquisition.positions.find_exterior_fovs` under the hood), the same definition `analysis/ffc.py`'s `"exterior_grid"` FFC-candidate strategy already uses. Used below only to bootstrap the background/foreground threshold (section 6) -- the real FFC field (section 5) is built from *interior* FOVs instead: a large fraction of "boundary" FOVs are not actually tissue-free (see the Review note in `01_elevation_heatmap.ipynb`).

In [ ]:
boundary_fov_ids, interior_fov_ids, grid_indices = identify_boundary_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
print(f"{len(boundary_fov_ids)} / {len(positions)} FOVs are boundary (exterior-grid) FOVs")
print(f"{len(interior_fov_ids)} / {len(positions)} FOVs are interior FOVs")

In [ ]:
half = config.step_size_um / config.non_overlap_fraction / 2   # true FOV footprint half-width

def plot_fov_highlight(highlight_ids, highlight_label, highlight_color, title, figure_name):
    fig, ax = plt.subplots(figsize=(8, 7))
    for fov_id, (x, y) in positions.items():
        is_hl = fov_id in highlight_ids
        ax.add_patch(mpatches.Rectangle(
            (x - half, y - half), 2 * half, 2 * half,
            lw=0.3, edgecolor=highlight_color if is_hl else "0.6",
            facecolor=highlight_color if is_hl else "0.6",
            alpha=0.6 if is_hl else 0.15,
        ))
    ax.plot([], [], "s", color=highlight_color, alpha=0.6, ms=8, label=f"{highlight_label} ({len(highlight_ids)})")
    ax.plot([], [], "s", color="0.6", alpha=0.3, ms=8, label=f"other ({len(positions) - len(highlight_ids)})")
    ax.invert_yaxis(); ax.axis("equal")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
    fig.tight_layout()
    fig.savefig(figures_dir / f"{figure_name}_round{target_round_id}.png", dpi=150)
    plt.show()

plot_fov_highlight(boundary_fov_ids, "boundary (exterior-grid)", "tab:red",
                    f"Boundary vs. interior FOVs -- round {target_round_id}", "tissue_thickness_boundary_fovs")

## 5 — FFC field: every interior FOV's own full-z-stack projection

Per `notebooks/tests/calculate_ffc/01_compare_ffc_methods.ipynb`'s own real comparison (median/max/min, smoothed/unsmoothed): **min projection** wins clearly -- a nucleus only occupies a handful of a FOV's z-planes at any given pixel, so the per-pixel minimum across the whole stack is overwhelmingly likely to be pure background, more robust to real tissue signal than the median or max. `FFC_METHOD` also accepts `"max"`, `"median"`, or `"mean"`.

**Reading every interior FOV's full z-stack serially can take hours** -- offers a SLURM array option (`cli_compute_fov_projections.py` + `cluster_submit.build_fov_projections_array_script`), one task per FOV, each reading only its own z-stack once. Re-run this cell later (after the job finishes) to pick up newly-written per-FOV projections; `calculate_ffc` builds the field once every interior FOV's is ready.

In [ ]:
projections_dir = cache_dir / "fov_projections" / f"round{target_round_id}_{int(CHANNEL_NM)}nm"
projections_dir.mkdir(parents=True, exist_ok=True)

interior_paths = {fov_id: projections_dir / f"fov{fov_id:04d}_{FFC_METHOD}.npy" for fov_id in interior_fov_ids}
to_compute = [f for f in interior_fov_ids if not interior_paths[f].exists()]
print(f"{len(interior_fov_ids) - len(to_compute)} / {len(interior_fov_ids)} interior FOV "
      f"'{FFC_METHOD}' projection(s) already cached; {len(to_compute)} more needed.")

USE_SLURM_ARRAY_FFC     = True   # set False to compute locally/serially instead (slow for many FOVs)
SLURM_ARRAY_CONCURRENCY_FFC = 50
SLURM_MEM_FFC           = "8gb"
SLURM_TIME_FFC          = "00:15:00"

if to_compute and USE_SLURM_ARRAY_FFC:
    from MERci.acquisition.cluster_submit import build_fov_projections_array_script, submit_sbatch, is_job_active

    ffc_job_sentinel = cache_dir / f"fov_projections_job_round{target_round_id}.json"
    cached_job = json.loads(ffc_job_sentinel.read_text()) if ffc_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"fov_projections_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"fov_projections_round{target_round_id}.sh"
        build_fov_projections_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=projections_dir,
            frame_indices=frame_idx_list, statistics=[FFC_METHOD], orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_FFC, mem=SLURM_MEM_FFC, time=SLURM_TIME_FFC,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            ffc_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label=f"Computing interior-FOV '{FFC_METHOD}' projections (local)")
    for fov_id in reporter.wrap(to_compute):
        img = compute_fov_projection(round_info.fov_files[fov_id][0], frame_idx_list, FFC_METHOD,
                                      orientation=MICROSCOPE_ORIENTATION)
        np.save(interior_paths[fov_id], img)

ffc_field_path = cache_dir / f"ffc_field_{FFC_METHOD}_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
if ffc_field_path.exists():
    ffc_field, ffc_meta = load_ffc_field(ffc_field_path)
    print(f"Loaded cached FFC field: {ffc_field_path}  ({ffc_meta})")
else:
    ffc_field, ffc_meta = calculate_ffc(
        sorted(interior_fov_ids), projections_dir, method=FFC_METHOD,
        smooth_sigma_px=FFC_SMOOTH_SIGMA_PX, normalize_percentile=FFC_NORMALIZE_PERCENTILE,
        ffc_min_value=FFC_MIN_VALUE,
    )
    if ffc_field is None:
        print(f"Still waiting on {len(ffc_meta['missing_fov_ids'])} interior FOV projection(s) -- "
              f"re-run this cell later once the SLURM array job above finishes.")
    else:
        save_ffc_field(ffc_field_path, ffc_field, {**ffc_meta, "round_id": target_round_id, "color": CHANNEL_NM})
        print(f"Computed + saved FFC field: {ffc_field_path}  ({ffc_meta})")

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet (see the cell above) -- skipping.")
else:
    cv = float(ffc_field.std() / ffc_field.mean())
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(ffc_field, cmap="viridis", vmin=0, vmax=1.05)
    ax.set_title(f"FFC field ({FFC_METHOD}, {ffc_meta['n_samples']} interior FOV(s))\n"
                 f"coeff. of variation = {cv:.3f}", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(figures_dir / f"tissue_thickness_ffc_field_round{target_round_id}.png", dpi=150)
    plt.show()

## 6 — Background/foreground threshold

`THRESHOLD` is auto-estimated as the highest pixel value (at `BACKGROUND_PERCENTILE`) observed among the `N_BACKGROUND_FRAMES` lowest-mean boundary FOVs' own mid-z frame, in the FFC-corrected + downsampled space thresholding actually happens in (`estimate_background_threshold`). Review the overlaid histogram plot before trusting the estimate -- if it looks wrong, set `THRESHOLD` by hand in section 2 and re-run from here.

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet (see section 5) -- skipping threshold estimation.")
else:
    boundary_ds_path = cache_dir / f"boundary_ds_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
    if boundary_ds_path.exists():
        _npz = np.load(boundary_ds_path)
        boundary_ds = {int(k.split("_")[1]): _npz[k] for k in _npz.files}
        print(f"Loaded {len(boundary_ds)} cached boundary-FOV downsampled frame(s): {boundary_ds_path}")
    else:
        boundary_ds = {}
        reporter = ProgressReporter(total=len(boundary_fov_ids), label="Reading+correcting boundary FOVs")
        for fov_id in reporter.wrap(sorted(boundary_fov_ids)):
            raw = read_image_frames(round_info.fov_files[fov_id][0], [mid_frame_idx])[0]
            boundary_ds[fov_id] = ffc_correct_and_downsample(
                raw, ffc_field, DOWNSAMPLE_FACTOR, MICROSCOPE_ORIENTATION,
            ).astype(np.float32)
        np.savez_compressed(boundary_ds_path, **{f"fov_{k}": v for k, v in boundary_ds.items()})
        print(f"Saved: {boundary_ds_path}")

    estimated_threshold = estimate_background_threshold(boundary_ds, N_BACKGROUND_FRAMES, BACKGROUND_PERCENTILE)
    print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of the {N_BACKGROUND_FRAMES} "
          f"lowest-mean boundary FOVs, FFC-corrected + downsampled space): {estimated_threshold:.1f}")
    if THRESHOLD is None:
        THRESHOLD = estimated_threshold
    print(f"Using THRESHOLD = {THRESHOLD:.1f}")

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet -- skipping.")
else:
    LOG_BINS = 200
    all_vals = np.concatenate([np.clip(ds, 1, None).ravel() for ds in boundary_ds.values()])
    bin_edges = np.linspace(np.log10(all_vals.min()), np.log10(all_vals.max()), LOG_BINS + 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    pooled_hist = np.zeros(LOG_BINS)
    for fov_id, ds in boundary_ds.items():
        hist, _ = np.histogram(np.log10(np.clip(ds, 1, None)), bins=bin_edges, density=True)
        ax.plot(bin_edges[:-1], hist, "-", lw=0.6, color="0.6", alpha=0.5)
        pooled_hist += hist
    pooled_hist /= len(boundary_ds)
    ax.plot(bin_edges[:-1], pooled_hist, "-", lw=2, color="k", label="pooled (mean over boundary FOVs)")
    ax.axvline(np.log10(max(THRESHOLD, 1)), color="tab:red", ls="--", label=f"THRESHOLD = {THRESHOLD:.0f}")
    ax.set_xlabel("log10(intensity)  [FFC-corrected, downsampled]", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("density", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"Boundary-FOV {CHANNEL_NM:.0f} nm intensity overlay ({len(boundary_ds)} FOVs)",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig.savefig(figures_dir / f"tissue_thickness_boundary_histograms_round{target_round_id}.png", dpi=150)
    plt.show()

## 7 — Per-FOV elevation matrices + downsampled z-stacks (full FOV grid)

For every real FOV in the round (not a representative subset) and every z-plane: FFC-correct, downsample, threshold, and record each pixel's topmost foreground z (`compute_fov_elevation`). Also caches the same z-stack's FFC-corrected, downsampled frames (reused directly by the single-z mosaic in section 9 and the z-sweep GIF in section 10 -- computed once here, never re-read).

**This is the heaviest read step in the notebook** (a full z-sweep per FOV, over the whole grid) -- offers the same SLURM array pattern as section 5 (`cli_compute_fov_elevation.py` + `cluster_submit.build_fov_elevation_array_script`), one task per FOV. Re-run this cell later (after the job finishes) to pick up newly-written results.

In [ ]:
elevation_dir = cache_dir / "elevation" / f"round{target_round_id}"
elevation_dir.mkdir(parents=True, exist_ok=True)

def elevation_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_elevation.npy"

def stack_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_stack.npy"

all_fov_ids = sorted(positions)   # every real FOV imaged in this round -- the full grid
to_compute = [f for f in all_fov_ids if not (elevation_path(f).exists() and stack_path(f).exists())]
print(f"{len(all_fov_ids) - len(to_compute)} / {len(all_fov_ids)} FOV(s) already cached; "
      f"{len(to_compute)} more needed ({len(z_frame_indices)} z-plane(s) each).")

USE_SLURM_ARRAY_ELEVATION     = True
SLURM_ARRAY_CONCURRENCY_ELEV  = 50
SLURM_MEM_ELEVATION           = "8gb"
SLURM_TIME_ELEVATION          = "00:15:00"

if ffc_field is None or THRESHOLD is None:
    print("FFC field/THRESHOLD not ready yet (see sections 5-6) -- skipping elevation computation.")
elif to_compute and USE_SLURM_ARRAY_ELEVATION:
    from MERci.acquisition.cluster_submit import build_fov_elevation_array_script, submit_sbatch, is_job_active

    elev_job_sentinel = cache_dir / f"elevation_job_round{target_round_id}.json"
    cached_job = json.loads(elev_job_sentinel.read_text()) if elev_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"elevation_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"elevation_round{target_round_id}.sh"
        build_fov_elevation_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=elevation_dir,
            frame_indices=frame_idx_list, z_um_values=z_um_list,
            ffc_field_path=ffc_field_path, threshold=THRESHOLD, downsample_factor=DOWNSAMPLE_FACTOR,
            orientation=MICROSCOPE_ORIENTATION, n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_ELEV, mem=SLURM_MEM_ELEVATION, time=SLURM_TIME_ELEVATION,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            elev_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV elevation matrices (local)")
    for fov_id in reporter.wrap(to_compute):
        M, ds_stack = compute_fov_elevation(
            round_info.fov_files[fov_id][0], frame_idx_list, z_um_list,
            ffc_field, THRESHOLD, DOWNSAMPLE_FACTOR, orientation=MICROSCOPE_ORIENTATION,
        )
        np.save(elevation_path(fov_id), M)
        np.save(stack_path(fov_id), ds_stack)

ready_fov_ids = [f for f in all_fov_ids if elevation_path(f).exists() and stack_path(f).exists()]
print(f"{len(ready_fov_ids)} / {len(all_fov_ids)} FOV(s) have both an elevation matrix and a stack ready.")
elevation_matrices = {f: np.load(elevation_path(f)) for f in ready_fov_ids}
stack_paths        = {f: stack_path(f) for f in ready_fov_ids}

In [ ]:
if not elevation_matrices:
    print("No FOV(s) ready yet.")
else:
    example_ids = sorted(elevation_matrices)[:6]
    fig, axes = plt.subplots(1, len(example_ids), figsize=(3 * len(example_ids), 3.2))
    vmax = max(float(elevation_matrices[f].max()) for f in example_ids)
    for ax, fov_id in zip(np.atleast_1d(axes), example_ids):
        im = ax.imshow(elevation_matrices[fov_id], cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(f"FOV {fov_id}", fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig.colorbar(im, ax=np.atleast_1d(axes).tolist(), shrink=0.8, label="elevation (um)")
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.suptitle("Example per-FOV elevation matrices (pre-crop)", fontsize=PLOT_TITLE_FONTSIZE)
    fig.savefig(figures_dir / f"tissue_thickness_example_fov_elevation_round{target_round_id}.png", dpi=150)
    plt.show()

## 8 — Per-FOV `z_last_um` + tissue thickness heatmap (full FOV grid)

Each FOV's scalar depth is just its own elevation matrix's maximum (0 = never foreground at any z -> `NaN`, no detected signal). Every FOV's elevation matrix is then cropped to its non-overlap footprint and stitched into one grid-indexed heatmap (`create_elevation_heatmap`) -- defaults to the full real FOV grid.

In [ ]:
results_rows = []
for fov_id in ready_fov_ids:
    z_max = float(elevation_matrices[fov_id].max())
    results_rows.append({
        "fov_id":    fov_id,
        "z_last_um": z_max if z_max > 0 else np.nan,
        "x_um":      meta.fovs[fov_id].position[0],
        "y_um":      meta.fovs[fov_id].position[1],
    })
results_df = pd.DataFrame(results_rows)

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Derived z_last_um for {len(results_df)} FOV(s); saved to {results_csv}")

In [ ]:
n_no_signal = int(results_df["z_last_um"].isna().sum())
print(f"{n_no_signal} / {len(results_df)} FOV(s) had no downsampled pixel above THRESHOLD at all "
      f"(no detected tissue signal).")
print("z_last_um:")
print(results_df["z_last_um"].describe())

In [ ]:
heatmap = create_elevation_heatmap(elevation_matrices, grid_indices, config, downsample_factor=DOWNSAMPLE_FACTOR)
np.save(cache_dir / f"elevation_heatmap_round{target_round_id}.npy", heatmap)
print(f"Stitched elevation heatmap shape: {heatmap.shape}")

In [ ]:
# Heatmap + colorbar + pixel-distribution panel, all three sharing the same
# displayed height via one shared axes divider: imshow's own aspect-equal
# scaling leaves the heatmap shorter than its subplot box (the stitched grid
# is much wider than tall), and fig.colorbar's default fraction/pad sizing
# doesn't track that actual displayed height -- appending both the colorbar
# and the distribution axes off the heatmap axes' own divider keeps all
# three locked to its real height regardless of the grid's aspect ratio.
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Zeros (background/no-signal pixels) vastly outnumber real tissue pixels and
# would otherwise swamp the distribution -- excluded from both the title
# statistic and the histogram below, not from the heatmap itself.
heatmap_values = heatmap[~np.isnan(heatmap)]
heatmap_values_nonzero = heatmap_values[heatmap_values != 0]

# "Full thickness" = this round's own actual imaging ceiling (the deepest
# z-plane any pixel could possibly reach), not MAX_Z_COLORMAP (just the
# colorbar's display cutoff -- close to it by construction, but not
# necessarily identical). A large fraction of pixels sitting exactly here
# means their real tissue signal extends AT LEAST this deep -- the imaged
# z-range simply didn't go any further, not that the tissue stops here.
full_thickness_um = float(heatmap_values_nonzero.max())
pct_full_thickness = 100.0 * np.mean(heatmap_values_nonzero == full_thickness_um)

fig, ax = plt.subplots(figsize=(13, 9))

cmap = plt.cm.turbo.copy(); cmap.set_bad("0.85")
im = ax.imshow(np.ma.masked_invalid(heatmap), cmap=cmap, vmin=0, vmax=MAX_Z_COLORMAP)
ax.set_title(f"Tissue thickness heatmap -- {len(ready_fov_ids)} FOV(s), "
             f"{pct_full_thickness:.1f}% pixels with thickness = {full_thickness_um:.1f} um",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.15)
cbar = fig.colorbar(im, cax=cax, label="thickness (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

# Bin edges at half-integers so each width-1 bin is centered on a whole
# micron (0.5, 1.5, ..., MAX_Z_COLORMAP + 0.5): the values themselves fall
# on a 0.5 um grid (this measurement's own z-step), so this groups each pair
# of adjacent half-micron levels into one integer-um bin. density=True + a
# log x-scale, since the bin at full_thickness_um is so tall on a linear
# scale (see the title's own percentage above) that every other bin was
# otherwise invisible next to it.
ax_hist = divider.append_axes("right", size="20%", pad=0.7)
HIST_BIN_EDGES = np.arange(0.5, MAX_Z_COLORMAP + 1.5, 1.0)
ax_hist.hist(heatmap_values_nonzero, bins=HIST_BIN_EDGES, orientation="horizontal",
             color="steelblue", density=True)
ax_hist.set_ylim(0, MAX_Z_COLORMAP)
ax_hist.yaxis.tick_right()
ax_hist.set_xscale("log")
ax_hist.xaxis.tick_top()
ax_hist.xaxis.set_label_position("top")
ax_hist.set_xlabel("density", fontsize=PLOT_LABEL_FONTSIZE)
ax_hist.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- tissue depth map", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

In [ ]:
# Contour-plot sibling of the heatmap above: same data, same colormap/value
# range, easier to trace where the tissue crosses a given depth than the
# heatmap's pixel-level texture. extend="max" fills every pixel above the
# top level (MAX_Z_COLORMAP) with the colormap's own top color instead of
# leaving it blank -- a large fraction of real pixels sit right at that
# ceiling (this round's own imaged z-limit), not just below it.
fig, ax = plt.subplots(figsize=(11, 9))
ax.set_facecolor("0.85")   # match the heatmap's own masked-region color

contour_levels = np.linspace(0, MAX_Z_COLORMAP, 21)
cs = ax.contourf(np.ma.masked_invalid(heatmap), levels=contour_levels,
                  cmap=plt.cm.turbo, vmin=0, vmax=MAX_Z_COLORMAP, extend="max")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title(f"Tissue thickness contour -- round {target_round_id}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.15)
cbar = fig.colorbar(cs, cax=cax, label="thickness (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.savefig(figures_dir / f"tissue_thickness_contour_round{target_round_id}.png", dpi=150)
plt.show()

## 9 — Mosaic at a single z (`Z_MOSAIC_UM`)

A static snapshot at one representative depth (default 25 um), from the same FFC-corrected, downsampled z-stacks section 7 already cached -- same shared-intensity-scale/z-label/scale-bar convention as the z-sweep GIF below (`create_z_mosaic`), just one frame saved as its own PNG rather than an animation. Useful as a standalone figure without opening the GIF.

In [ ]:
z_mosaic_path = figures_dir / f"tissue_thickness_z_mosaic_round{target_round_id}.png"
create_z_mosaic(
    stack_paths, z_um_list, grid_indices, config, z_mosaic_path, z_um=Z_MOSAIC_UM,
    downsample_factor=DOWNSAMPLE_FACTOR, scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP,
)
print(f"Saved: {z_mosaic_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.array(Image.open(z_mosaic_path).convert("L")), cmap="gray")
ax.set_title(f"Mosaic at z ~ {Z_MOSAIC_UM:.1f} um -- round {target_round_id}", fontsize=PLOT_TITLE_FONTSIZE)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 10 — z-sweep GIF

Same FFC-corrected, downsampled z-stacks section 7 already cached, stitched per z-plane into one GIF (`create_gif`) -- a shared intensity scale across every frame (so brightness changes reflect real signal fading, not per-frame auto-contrast), a per-frame `"z = <value> um"` label, and a physical scale bar (`GIF_SCALEBAR_UM`, default 1000 um -> "1 mm").

Each rendered frame is also cached to disk (`analysis/cache/measure_tissue_thickness/gif_frames/round<N>/`), so if the cell crashes during the final GIF encode/save (the slow full-grid stitching loop -- the part the progress bar tracks -- is already done by then), re-running only redoes the encode step instead of every frame.


In [ ]:
gif_path = figures_dir / f"tissue_elevation_gif_round{target_round_id}.gif"
gif_frame_cache_dir = cache_dir / "gif_frames" / f"round{target_round_id}"
n_cached_frames = len(list(gif_frame_cache_dir.glob("z*.png"))) if gif_frame_cache_dir.exists() else 0
print(f"{n_cached_frames} GIF frame(s) already cached in {gif_frame_cache_dir}")
create_gif(
    stack_paths, z_um_list, grid_indices, config, gif_path,
    downsample_factor=DOWNSAMPLE_FACTOR, z_stride=GIF_Z_STRIDE, frame_duration_ms=GIF_FRAME_DURATION_MS,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP, frame_cache_dir=gif_frame_cache_dir,
)
print(f"Saved: {gif_path}")

In [ ]:
with Image.open(gif_path) as im:
    n_gif_frames = im.n_frames
    first_frame = np.array(im.convert("L"))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(first_frame, cmap="gray")
ax.set_title(f"First GIF frame ({n_gif_frames} frame(s) total) -- open {gif_path.name} to view the full sweep",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 11 — z-sweep movie (MP4, for PowerPoint)

Same frames as section 10 -- `create_movie` reuses that GIF's own `gif_frame_cache_dir`, so nothing gets re-rendered here. Written as H.264 `.mp4` (silent) instead of a GIF: PowerPoint (Windows and Mac) inserts an `.mp4` as a real video object ("Insert > Video"), where it only ever treats a GIF as a static/animated picture. Starts on the same first frame already previewed above.


In [ ]:
movie_path = figures_dir / f"tissue_elevation_movie_round{target_round_id}.mp4"
create_movie(
    stack_paths, z_um_list, grid_indices, config, movie_path,
    downsample_factor=DOWNSAMPLE_FACTOR, z_stride=GIF_Z_STRIDE, frame_duration_ms=GIF_FRAME_DURATION_MS,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP, frame_cache_dir=gif_frame_cache_dir,
)
print(f"Saved: {movie_path}")